In [1]:
# %%
import torch
import numpy as np
import matplotlib.pyplot as plt

# read data
import sys
sys.path.append('../../..')
import data_preprocessing
from importlib import reload
import importlib

# training process
import algorithm.algo2 as algo2

# evaluation
import algorithm.kmeans_gpu as kmeans
reload(kmeans)
import eval_metrics.ac
import eval_metrics.nmi

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

[Using device] cuda


In [ ]:

# %%
import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--dataset', type=str, help='dataset name')
parser.add_argument('--n_clusters', type=int, help='number of clusters')
parser.add_argument('--classes', type=str, default=None, help='classes to use')
parser.add_argument('--filename', type=str, help='filename to save the result')

args = parser.parse_args()

print(f'[args] {args}')

f = open(args.filename, 'a')

# %%
# DATASET = getattr(data_preprocessing, args.dataset)
DATASET = importlib.import_module(f'data_preprocessing.{args.dataset}')
CLUSTER_NUM = args.n_clusters
CLASS_LIST = eval(args.classes) if args.classes is not None else None
if CLASS_LIST is not None:
    CLASS_LIST.sort()
    assert len(CLASS_LIST) == CLUSTER_NUM, 'The length of class list should be equal to n_clusters'

In [2]:
from data_preprocessing import tdt2
DATASET = tdt2
CLUSTER_NUM = 20

In [7]:
# %%
reload(kmeans)
# read data
mat, label = DATASET.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, 
                              random_choose_class=True)
mat = torch.tensor(mat).to(device)
label = torch.tensor(label).to(device)

print('[Data shape]', mat.shape)
print('[Label]', torch.unique(label, return_counts=False))

with torch.no_grad():
    pred_labels, _ = kmeans.kmeans(mat, CLUSTER_NUM)

[Data shape] torch.Size([5200, 36771])
[Label] tensor([ 1,  4,  5,  6,  7, 12, 13, 14, 15, 17, 18, 19, 21, 23, 24, 25, 26, 27,
        29, 30], device='cuda:0', dtype=torch.uint8)


OutOfMemoryError: CUDA out of memory. Tried to allocate 28.49 GiB. GPU 0 has a total capacty of 23.64 GiB of which 19.76 GiB is free. Process 324232 has 3.88 GiB memory in use. Of the allocated memory 3.43 GiB is allocated by PyTorch, and 12.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [28]:
# %%
label_eval = label.cpu().numpy()
pred_labels_eval = pred_labels.cpu().numpy()

acc = eval_metrics.ac.ac(label_eval, pred_labels_eval)
nmi = eval_metrics.nmi.nmi(label_eval, pred_labels_eval)
print(f'[Accuracy] {acc}\n[NMI] {nmi}')
print()


# print(f"{args.dataset},{CLUSTER_NUM},{'/'.join(list(map(str, np.unique(np.array(label)).tolist())))},{TOL},{acc},{nmi}", file=f)

# f.close()

# %%





[Accuracy] 0.3135678391959799
[NMI] 0.2359170635509634

